# Ensemble Methods: Combining Models for Better PredictionsA single model often makes mistakes — but a *crowd* of models, when combined wisely, can dramatically reduce error and improve generalization. This notebook explores the major ensemble techniques in machine learning.

## 1. The Wisdom of the Crowd — Why Ensembles WorkThe fundamental insight behind ensemble methods is the **bias-variance decomposition** of the generalization error:$$\text{Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Error}$$- **Bias**: Error from oversimplifying the true relationship (underfitting)- **Variance**: Error from being too sensitive to training data fluctuations (overfitting)Ensembles reduce error by averaging out variance (bagging) or by sequentially reducing bias (boosting).> *"A model that is too complex overfits; a model too simple underfits. An ensemble balances the two."*

## 2. Types of Ensembles| Method | Strategy | Key Idea ||--------|----------|----------|| **Bagging** | Parallel training | Train models independently on bootstrap samples, average predictions || **Boosting** | Sequential training | Train models one after another, each correcting predecessor's errors || **Stacking** | Meta-learning | Train a "meta-model" on predictions of base models |

In [ ]:
# ===== IMPORTS =====
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

from sklearn.datasets import make_classification, make_moons
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    VotingClassifier,
    StackingClassifier,
    RandomForestRegressor,
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.inspection import partial_dependence, PartialDependenceDisplay

sns.set_theme(style='whitegrid')
np.random.seed(42)
print('All imports successful.')

---## 3. Voting Classifiers: Hard vs Soft VotingThe simplest ensemble: train several independent classifiers and let them **vote** on the prediction.- **Hard voting**: each model casts one vote, majority wins.- **Soft voting**: each model outputs class probabilities; probabilities are averaged, highest average wins.

In [ ]:
# Create a synthetic dataset
X, y = make_classification(
    n_samples=1000, n_features=10, n_informative=8,
    n_redundant=0, n_clusters_per_class=2, flip_y=0.05, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f'Train: {X_train.shape[0]} samples, Test: {X_test.shape[0]} samples')

In [ ]:
# Individual classifiers
clf1 = LogisticRegression(max_iter=500, random_state=42)
clf2 = DecisionTreeClassifier(max_depth=5, random_state=42)
clf3 = KNeighborsClassifier(n_neighbors=7)

# Hard voting
voting_hard = VotingClassifier(
    estimators=[('lr', clf1), ('dt', clf2), ('knn', clf3)],
    voting='hard'
)
# Soft voting
voting_soft = VotingClassifier(
    estimators=[('lr', clf1), ('dt', clf2), ('knn', clf3)],
    voting='soft'
)

for name, clf in [('Logistic Regression', clf1),
                   ('Decision Tree', clf2),
                   ('KNN', clf3),
                   ('Hard Voting', voting_hard),
                   ('Soft Voting', voting_soft)]:
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    print(f'{name:25s}  Accuracy: {acc:.4f}')

---## 4. Bagging: Bootstrap AggregationBagging trains the same base estimator on **bootstrap samples** (random sampling with replacement) of the training data, then averages predictions. This primarily reduces **variance** without increasing bias.

In [ ]:
# BaggingClassifier with a Decision Tree base
base_dt = DecisionTreeClassifier(random_state=42)

bag_clf = BaggingClassifier(
    estimator=base_dt,
    n_estimators=300,
    max_samples=1.0,
    bootstrap=True,
    random_state=42,
    n_jobs=-1,
    oob_score=True
)
bag_clf.fit(X_train, y_train)
y_pred_bag = bag_clf.predict(X_test)

single_dt = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
y_pred_dt = single_dt.predict(X_test)

print(f'Single Decision Tree Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}')
print(f'Bagging (300 trees)  Accuracy: {accuracy_score(y_test, y_pred_bag):.4f}')
print(f'Out-of-Bag Score:              {bag_clf.oob_score_:.4f}')

### 4.1 Out-of-Bag (OOB) EvaluationFor each bootstrap sample, about 63% of the original instances are sampled; the remaining 37% are **out-of-bag**. These out-of-bag samples can serve as a built-in validation set — no need for a separate validation split (or cross-validation) to estimate generalization error.

In [ ]:
# Visualize OOB error vs number of trees
oob_errors = []
n_trees_range = range(10, 310, 10)

for n in n_trees_range:
    bc = BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=42),
        n_estimators=n, oob_score=True, random_state=42, n_jobs=-1
    )
    bc.fit(X_train, y_train)
    oob_errors.append(1 - bc.oob_score_)

plt.figure(figsize=(10, 5))
plt.plot(n_trees_range, oob_errors, 'b-o', markersize=4)
plt.xlabel('Number of trees')
plt.ylabel('OOB Error')
plt.title('OOB Error vs Number of Trees (Bagging)')
plt.grid(True)
plt.tight_layout()
plt.show()

### 4.2 Random Forests — Bagging + Feature Randomness**Random Forests** extend bagging by adding feature randomness: at each split, only a random subset of features is considered (`max_features`). This decorrelates the trees further, leading to even lower variance.**Extremely Randomized Trees (ExtraTrees)** go a step further: instead of searching for the best split threshold, they pick a **random split threshold** for each candidate feature. This adds even more randomness and often trains faster.

In [ ]:
# Compare Random Forest vs ExtraTrees
rf = RandomForestClassifier(
    n_estimators=200, max_depth=12,
    min_samples_leaf=2, random_state=42, n_jobs=-1
)
et = ExtraTreesClassifier(
    n_estimators=200, max_depth=12,
    min_samples_leaf=2, random_state=42, n_jobs=-1
)

rf.fit(X_train, y_train)
et.fit(X_train, y_train)

print(f'RandomForest Accuracy:  {accuracy_score(y_test, rf.predict(X_test)):.4f}')
print(f'ExtraTrees Accuracy:    {accuracy_score(y_test, et.predict(X_test)):.4f}')

---## 5. BoostingBoosting trains models **sequentially**, where each new model focuses on the mistakes of the previous ones. This primarily reduces **bias**.

### 5.1 AdaBoost (Adaptive Boosting)AdaBoost assigns **weights** to training instances. Misclassified samples get higher weights, so subsequent models focus more on hard cases.

In [ ]:
# AdaBoost
ada_clf = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
    n_estimators=200,
    learning_rate=0.5,
    random_state=42
)
ada_clf.fit(X_train, y_train)
y_pred_ada = ada_clf.predict(X_test)
print(f'AdaBoost Accuracy: {accuracy_score(y_test, y_pred_ada):.4f}')

### 5.2 Gradient BoostingInstead of re-weighting samples, Gradient Boosting fits each new tree to the **residual errors** of the current ensemble. The key hyperparameters:- `n_estimators`: number of boosting stages- `learning_rate`: shrinks the contribution of each tree (lower = better generalization)- `subsample`: fraction of samples used per tree (stochastic GBDT)- `max_depth`: depth of each tree (usually 3–5)

In [ ]:
# Gradient Boosting
gb_clf = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    random_state=42
)
gb_clf.fit(X_train, y_train)
y_pred_gb = gb_clf.predict(X_test)
print(f'Gradient Boosting Accuracy: {accuracy_score(y_test, y_pred_gb):.4f}')

In [ ]:
# Learning curves for Gradient Boosting (training & validation accuracy vs n_estimators)
gb_staged = GradientBoostingClassifier(
    n_estimators=300, learning_rate=0.1, max_depth=3,
    subsample=0.8, random_state=42
)
gb_staged.fit(X_train, y_train)

# Staged predictions
train_scores = []
test_scores = []
for train_pred in gb_staged.staged_predict(X_train):
    train_scores.append(accuracy_score(y_train, train_pred))
for test_pred in gb_staged.staged_predict(X_test):
    test_scores.append(accuracy_score(y_test, test_pred))

plt.figure(figsize=(10, 5))
plt.plot(range(1, 301), train_scores, label='Training Accuracy', alpha=0.8)
plt.plot(range(1, 301), test_scores, label='Test Accuracy', alpha=0.8)
plt.xlabel('Number of Estimators')
plt.ylabel('Accuracy')
plt.title('Gradient Boosting: Learning Curve')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Early stopping with validation fraction
gb_early = GradientBoostingClassifier(
    n_estimators=1000,
    learning_rate=0.1,
    max_depth=3,
    validation_fraction=0.2,
    n_iter_no_change=10,
    random_state=42
)
gb_early.fit(X_train, y_train)
print(f'Gradient Boosting (early stopping) used {gb_early.n_estimators_} estimators')
print(f'Accuracy: {accuracy_score(y_test, gb_early.predict(X_test)):.4f}')

### 5.3 XGBoost — A Quick Overview**XGBoost** (eXtreme Gradient Boosting) is a highly optimized gradient boosting library that dominates tabular-data competitions. Key innovations:- **Regularized boosting**: adds L1/L2 penalties on leaf weights → reduces overfitting- **Column subsampling**: like Random Forests, randomly selects features per tree/split- **Weighted quantile sketch**: efficient handling of weighted data- **Cache-aware & out-of-core computation**: extremely fast training> *XGBoost is not installed in this environment, but its concepts are well captured by sklearn's `GradientBoostingClassifier` with tuned regularisation.*

---## 6. Stacking / Stacked Generalization**Stacking** trains a meta-learner on the predictions of multiple base models.- **Blending**: holdout set for meta-features (simple, prone to overfitting).- **Stacking (with CV)**: k-fold cross-validation to generate meta-features — each base model's out-of-fold predictions become the training data for the meta-learner. More robust.

In [ ]:
# StackingClassifier with CV
base_learners = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('svm', SVC(probability=True, random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=7)),
    ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42))
]
meta_learner = LogisticRegression(max_iter=500, random_state=42)

stack_clf = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    cv=5
)
stack_clf.fit(X_train, y_train)
y_pred_stack = stack_clf.predict(X_test)
print(f'Stacking Accuracy: {accuracy_score(y_test, y_pred_stack):.4f}')

---## 7. Comparison of All Ensemble MethodsLet's compare 7 models on the same classification dataset: Single Decision Tree, Bagging, Random Forest, ExtraTrees, AdaBoost, Gradient Boosting, and Stacking.

In [ ]:
# Train all 7 models
models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=8, random_state=42),
    'Bagging (DT)': BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=42),
        n_estimators=200, random_state=42, n_jobs=-1
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=12, random_state=42, n_jobs=-1
    ),
    'ExtraTrees': ExtraTreesClassifier(
        n_estimators=200, max_depth=12, random_state=42, n_jobs=-1
    ),
    'AdaBoost': AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
        n_estimators=200, learning_rate=0.5, random_state=42
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42
    ),
    'Stacking': StackingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
            ('svm', SVC(probability=True, random_state=42)),
            ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42))
        ],
        final_estimator=LogisticRegression(max_iter=500, random_state=42),
        cv=5
    )
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = accuracy_score(y_test, y_pred)
    print(f'{name:20s}  Accuracy: {results[name]:.4f}')

In [ ]:
# VISUALIZATION 1: Accuracy comparison bar chart
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2', '#CCB974', '#64B5CD', '#E47A9C']

plt.figure(figsize=(12, 6))
bars = plt.bar(results.keys(), results.values(), color=colors, edgecolor='black')

for bar, val in zip(bars, results.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.ylim(0.85, 1.0)
plt.ylabel('Accuracy')
plt.title('Model Comparison: Accuracy on Test Set')
plt.xticks(rotation=15, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---## 8. Feature Importance from Random Forest and Gradient BoostingBoth Random Forest and Gradient Boosting provide a `feature_importances_` attribute, which measures how much each feature contributes to reducing impurity across all trees.

In [ ]:
# VISUALIZATION 2: Feature importance comparison
feature_names = [f'Feature {i}' for i in range(X.shape[1])]

rf_importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)
gb_importances = pd.Series(gb_clf.feature_importances_, index=feature_names).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rf_importances.plot(kind='bar', ax=axes[0], color='#55A868', edgecolor='black')
axes[0].set_title('Feature Importance — Random Forest')
axes[0].set_ylabel('Importance')
axes[0].tick_params(axis='x', rotation=45)

gb_importances.plot(kind='bar', ax=axes[1], color='#4C72B0', edgecolor='black')
axes[1].set_title('Feature Importance — Gradient Boosting')
axes[1].set_ylabel('Importance')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---## 9. Partial Dependence PlotsPartial dependence plots (PDPs) show how a feature affects the model's predictions **on average**, marginalizing over all other features. This helps interpret complex ensemble models.

In [ ]:
# VISUALIZATION 3: Partial dependence plots for Gradient Boosting
top_features = gb_importances.head(4).index.tolist()
feature_indices = [list(feature_names).index(f) for f in top_features]

fig, ax = plt.subplots(figsize=(12, 8))
display = PartialDependenceDisplay.from_estimator(
    gb_clf, X_train, feature_indices,
    feature_names=feature_names,
    grid_resolution=50, ax=ax
)
fig.suptitle('Partial Dependence Plots — Gradient Boosting', fontsize=14)
plt.tight_layout()
plt.show()

---## 10. Decision Boundary VisualizationTo build intuition, let's visualize how different ensemble methods partition the feature space on a 2D problem.

In [ ]:
# VISUALIZATION 4: Decision boundaries on moons dataset
X_moon, y_moon = make_moons(n_samples=500, noise=0.3, random_state=42)
Xm_train, Xm_test, ym_train, ym_test = train_test_split(X_moon, y_moon, test_size=0.3, random_state=42)

models_2d = {
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'AdaBoost': AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
        n_estimators=100, random_state=42
    )
}

xx, yy = np.meshgrid(np.linspace(X_moon[:, 0].min() - 0.5, X_moon[:, 0].max() + 0.5, 200),
                     np.linspace(X_moon[:, 1].min() - 0.5, X_moon[:, 1].max() + 0.5, 200))

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, (name, model) in zip(axes, models_2d.items()):
    model.fit(Xm_train, ym_train)
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(Xm_train[:, 0], Xm_train[:, 1], c=ym_train, cmap='coolwarm',
               edgecolor='black', s=30, alpha=0.8)
    acc = accuracy_score(ym_test, model.predict(Xm_test))
    ax.set_title(f'{name} (Acc: {acc:.3f})')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

In [ ]:
# VISUALIZATION 5: OOB error vs trees for Random Forest
n_trees = range(10, 310, 10)
rf_oob_errors = []

for n in n_trees:
    rf_oob = RandomForestClassifier(
        n_estimators=n, oob_score=True, random_state=42, n_jobs=-1
    )
    rf_oob.fit(X_train, y_train)
    rf_oob_errors.append(1 - rf_oob.oob_score_)

plt.figure(figsize=(10, 5))
plt.plot(n_trees, rf_oob_errors, 'g-s', markersize=4, label='Random Forest (OOB)')
plt.xlabel('Number of trees')
plt.ylabel('OOB Error')
plt.title('OOB Error vs Number of Trees — Random Forest')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

---## 11. When to Use Which Ensemble Method| Situation | Recommended Method | Why ||-----------|-------------------|-----|| **High-variance model** (deep trees) | **Bagging / Random Forest** | Averaging reduces variance without increasing bias || **High-bias model** (shallow trees / stumps) | **Boosting (GBDT, XGBoost)** | Sequential correction reduces bias || **Noisy data / outliers** | **Random Forest** | Robust to outliers (averaging); boosting can overfit noise || **Large tabular dataset** | **XGBoost / LightGBM** | Best speed-accuracy tradeoff; GPU support || **Heterogeneous data** (text + images + tabular) | **Stacking** | Combine diverse model types (NN, Tree, Linear) || **Maximum accuracy, unlimited compute** | **Stacking + tuned base models** | Meta-learning squeezes out extra performance || **Interpretability needed** | **Random Forest** (impurity) or **single DT** | Feature importance; can inspect individual trees || **Very fast training needed** | **ExtraTrees** | Random split thresholds make training 2-5x faster than RF || **Class imbalance** | **Boosting (with class weights)** | Iterative focus on hard (minority) samples helps |

---## 12. Practice ExercisesTest your understanding with these exercises:### Exercise 1: Voting Threshold
Modify the `VotingClassifier` so that instead of majority vote, a prediction is only made when **at least 2 out of 3** classifiers agree. For ties, use the prediction of a designated "tiebreaker" model. Implement and test this custom voting scheme.### Exercise 2: OOB vs CV
On a dataset of your choice, compare the OOB score of a `RandomForestClassifier` (n_estimators=500) with 5-fold cross-validation accuracy. Are they consistently close? What happens if you reduce `max_samples` in a `BaggingClassifier`?### Exercise 3: Boosting Hyperparameter Sensitivity
Systematically vary `learning_rate` (0.001, 0.01, 0.1, 0.5, 1.0) and `n_estimators` (50, 100, 200, 500) for `GradientBoostingClassifier`. Create a heatmap of test accuracy to visualize the interaction. Where is the sweet spot?### Exercise 4: Stacking Without Sklearn
Implement a manual stacking ensemble: train 3 base models on the training set, create a meta-feature matrix using 5-fold cross-validated predictions, then train a logistic regression meta-model. Compare performance to sklearn's `StackingClassifier`.### Exercise 5: Feature Importance Stability
Run Random Forest 10 times (different `random_state`) on the same dataset and collect feature importances each time. Compute the mean and standard deviation for each feature. Which features are consistently important? Create an error-bar plot showing mean ± std feature importance.